# Chapter 6 — Algorithm Chains and Pipelines

## Learning Objectives

By the end of this notebook, you should be able to:

1. Explain why pipelines are important.
2. Build pipelines that combine preprocessing and modeling.
3. Use grid search with pipeline parameters.
4. Add feature selection inside a pipeline.
5. Understand how pipelines prevent data leakage.

## Chapter Overview

A machine learning workflow often includes multiple steps: scaling, feature selection, transformation, and modeling. If these steps are performed manually, it is easy to accidentally leak information from the test set into training.

`Pipeline` solves this by treating the full workflow as a single estimator.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Dataset Preparation

We use the Breast Cancer dataset because it contains numerical features and a binary classification target. The dataset is split before building any model.

In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=RANDOM_STATE
)

print('Training shape:', X_train.shape)
print('Test shape:', X_test.shape)

Training shape: (426, 30)
Test shape: (143, 30)


## 2. Basic Pipeline

A basic pipeline can combine scaling and classification.

### Theory

SVM is sensitive to feature scale. If features have very different magnitudes, the distance-based kernel calculation can be dominated by features with larger numeric ranges. Standardization helps each feature contribute more fairly.

In [3]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC())
])

pipe.fit(X_train, y_train)
print('Pipeline test accuracy:', round(pipe.score(X_test, y_test), 4))

Pipeline test accuracy: 0.979


### Output Interpretation

The pipeline automatically applies scaling before fitting the SVM. During prediction, the same scaling transformation is applied to the test data.

## 3. Grid Search with Pipeline

Pipeline parameters are accessed using the format:

`step_name__parameter_name`

For example, `svm__C` means the `C` parameter inside the `svm` step.

In [4]:
param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': [0.001, 0.01, 0.1]
}

grid = GridSearchCV(pipe, param_grid=param_grid, cv=5)
grid.fit(X_train, y_train)

print('Best parameters:', grid.best_params_)
print('Best CV score:', round(grid.best_score_, 4))
print('Test score:', round(grid.score(X_test, y_test), 4))

Best parameters: {'svm__C': 10, 'svm__gamma': 0.001}
Best CV score: 0.9765
Test score: 0.979


### Output Interpretation

The scaler is fitted separately inside each cross-validation fold. This is the correct procedure because validation data should not influence preprocessing parameters.

## 4. Pipeline with Feature Selection

Feature selection can also be part of a pipeline. This prevents the selected features from being chosen using information from validation or test data.

In [5]:
pipe_select = Pipeline([
    ('scaler', StandardScaler()),
    ('select', SelectPercentile(score_func=f_classif)),
    ('logreg', LogisticRegression(max_iter=5000))
])

param_grid = {
    'select__percentile': [30, 50, 70, 100],
    'logreg__C': [0.1, 1, 10]
}

grid_select = GridSearchCV(pipe_select, param_grid=param_grid, cv=5)
grid_select.fit(X_train, y_train)

print('Best parameters:', grid_select.best_params_)
print('Best CV score:', round(grid_select.best_score_, 4))
print('Test score:', round(grid_select.score(X_test, y_test), 4))

Best parameters: {'logreg__C': 0.1, 'select__percentile': 100}
Best CV score: 0.9765
Test score: 0.979


### Output Interpretation

The grid search chooses both the number of selected features and the logistic regression regularization strength. Because everything is inside the pipeline, feature selection is performed correctly within each training fold.

## 5. Comparing Models with Consistent Workflows

Pipelines make experiments cleaner. Different models can be compared under consistent preprocessing rules.

In [6]:
model_candidates = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    'SVM': make_pipeline(StandardScaler(), SVC(C=10, gamma=0.01)),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
}

for name, model in model_candidates.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=5)
    model.fit(X_train, y_train)
    print(f'{name:20s} | CV mean={cv_scores.mean():.3f} | Test={model.score(X_test, y_test):.3f}')

Logistic Regression  | CV mean=0.969 | Test=0.986
SVM                  | CV mean=0.962 | Test=0.979


Random Forest        | CV mean=0.960 | Test=0.958


### Output Interpretation

The comparison gives a quick view of which approach performs best. Random Forest does not require scaling, while Logistic Regression and SVM benefit from scaling.

## Additional Notes: Data Leakage and Reproducibility

A common mistake in machine learning is applying preprocessing before splitting or before cross-validation. For example, if a scaler is fitted on the full dataset before validation, information from the validation fold influences the transformation. This is called **data leakage**. The result can look better than the model actually is.

Pipelines also improve reproducibility because the exact order of operations is stored in one object. This makes experiments easier to review, rerun, and share. In academic work, this is important because the submitted notebook should clearly show how data is transformed and how the model is trained.

When a workflow contains feature selection, dimensionality reduction, scaling, and modeling, all steps that learn from data should be inside the pipeline. This ensures that every cross-validation fold behaves like a realistic future deployment scenario.


## Key Takeaways

- Pipelines combine preprocessing and modeling into a single workflow.
- Pipelines reduce the risk of data leakage.
- Grid search can tune parameters inside pipeline steps.
- Feature selection should be inside the pipeline when using cross-validation.
- Pipelines make experiments more reproducible and organized.

## Chapter Summary

Chapter 6 shows that a reliable machine learning workflow requires more than model fitting. The full chain of preprocessing, transformation, feature selection, and modeling must be evaluated correctly.